<a href="https://colab.research.google.com/github/crezny/DATASCI266_final_project/blob/main/KG Construction/2-Ontologies to Neo4j.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Loads ontologies from DB into Neo4j

In [ ]:
%pip install neo4j

In [ ]:
from neo4j import GraphDatabase
from pathlib import Path
import json, os
from datetime import datetime
from dotenv import load_dotenv
import pandas as pd

import ast

load_dotenv()  # read NEO4J_URI, NEO4J_USER, NEO4J_PASS from .env

import os
try:
    from google.colab import userdata
    get_secret = userdata.get
except ImportError:
    # Fallback: define a dummy get_secret for local use
    def get_secret(key):
        return None

from dotenv import load_dotenv
load_dotenv()


from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError

def get_env(key: str, default: str = None):
    """Tries os.getenv first, then Colab userdata if available."""
    return os.getenv(key) or get_secret(key) or default

NEO4J_URI  = get_env("NEO4J_URI")   # bolt:// or neo4j://
NEO4J_USER = get_env("NEO4J_USER")
NEO4J_PASS = get_env("NEO4J_PASS")
NEO4J_DATABASE = get_env("NEO4J_DATABASE")

DB_HOST = get_env("POSTGRES_DB_HOST")
DB_PORT = int(get_env("POSTGRES_DB_PORT"))
DB_NAME = get_env("POSTGRES_DB_NAME")
DB_USER = get_env("POSTGRES_DB_USER")
DB_PASS = get_env("POSTGRES_DB_PASS")

connection_url = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"


In [ ]:
def parse_set_like(value):
    """Convert stringified sets like "{'a','b'}" into lists."""
    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, (set, list, tuple)):
                return list(parsed)
        except (ValueError, SyntaxError):
            pass
        # fallback: split by comma if not proper set
        return [v.strip().strip("'\"") for v in value.strip("{}").split(",") if v.strip()]
    elif isinstance(value, (set, list, tuple)):
        return list(value)
    return []

def get_ontology_from_db():
    engine = create_engine(connection_url, echo=False)
    db_ontology = pd.read_sql('SELECT * FROM ontology;', engine)
    result = {
        row["id"]: {
            "id": row["id"],
            "label": row["label"],
            "description": row["description"],
            "category": row["category"],
            "seeds": parse_set_like(row["seeds"]),
            "counterexamples": parse_set_like(row["counterexamples"]),
            "examples": parse_set_like(row["examples"]),
            "required": bool(row["required"]),
        }
        for _, row in db_ontology.iterrows()
    }
    return result
ontology = get_ontology_from_db()
len(ontology), list(ontology.keys())

In [ ]:
def category_id(label: str) -> str:
    slug = label.strip().lower().replace(" ", "_")
    return f"cat.{slug}"

# Collect categories from the ontology entries
categories = sorted({ entry.get("category", "Uncategorized") for entry in ontology.values() })
categories_map = { c: category_id(c) for c in categories }
categories_map


In [ ]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))

def upsert_categories(session, categories_map: dict):
    cypher = """
    UNWIND $rows AS row
    MERGE (c:Category {category_id: row.category_id})
      ON CREATE SET c.label = row.label, c.createdAt = datetime()
      ON MATCH  SET c.label = row.label
    """
    rows = [{"category_id": cid, "label": label} for label, cid in categories_map.items()]
    session.run(cypher, rows=rows)

def upsert_ontology_nodes(session, ontology_dict: dict, categories_map: dict):
    """
    Creates/updates :Ontology nodes and BELONGS_TO relationships to :Category.
    Assumes entries like:
      {
        "controls.authorization": {
          "id": "controls.authorization",
          "label": "Authorization Controls",
          "description": "...",
          "category": "Controls",
          "seeds": ["least privilege", ...],
          "counterexamples": ["authentication", ...],
          "required": true/false (optional)
        }
      }
    """
    rows = []
    now = datetime.utcnow().isoformat()

    for key, node in ontology_dict.items():
        node_id = node.get("id") or key
        label = node.get("label") or key.split(".")[-1].replace("_", " ").title()
        desc  = node.get("description") or node.get("desc", "")
        cat_label = node.get("category", "Uncategorized")
        cat_id = categories_map.get(cat_label, category_id(cat_label))
        seeds = node.get("seeds", []) or []
        ctrs  = node.get("counterexamples", []) or []
        required = bool(node.get("required", False))

        rows.append({
            "node_id": node_id,
            "label": label,
            "description": desc,
            "category_id": cat_id,
            "required": required,
            "seeds": seeds,
            "counterexamples": ctrs,
            "now": now
        })

    cypher = """
    // Upsert Ontology nodes
    UNWIND $rows AS row
    MERGE (n:Ontology {node_id: row.node_id})
      ON CREATE SET
        n.label = row.label,
        n.description = row.description,
        n.required = row.required,
        n.seeds = row.seeds,
        n.counterexamples = row.counterexamples,
        n.createdAt = datetime(row.now)
      ON MATCH SET
        n.label = row.label,
        n.description = row.description,
        n.required = row.required,
        n.seeds = row.seeds,
        n.counterexamples = row.counterexamples

    // Upsert Category and relationship
    WITH row, n
    MERGE (c:Category {category_id: row.category_id})
      ON CREATE SET c.label = split(row.category_id, '.')[1], c.createdAt = datetime(row.now)
    MERGE (n)-[:BELONGS_TO]->(c)
    """
    session.run(cypher, rows=rows)


In [ ]:
with driver.session(database=NEO4J_DATABASE) as session:
    upsert_categories(session, categories_map)
    upsert_ontology_nodes(session, ontology, categories_map)

print("Ontology loaded into Neo4j.")


In [ ]:
with driver.session(database=NEO4J_DATABASE) as session:
    print("Ontology nodes:", session.run("MATCH (n:Ontology) RETURN count(n) AS c").single()["c"])
    print("Categories:",     session.run("MATCH (c:Category) RETURN count(c) AS c").single()["c"])
    sample = session.run("""
        MATCH (n:Ontology)-[:BELONGS_TO]->(c:Category)
        RETURN n.node_id AS id, n.label AS label, c.label AS category
        LIMIT 10
    """).data()
sample
